# SynthID-Text

Short notebook to test synthid-text with llama3.1-8B

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import LogitsProcessorList
from transformers import SynthIDTextWatermarkLogitsProcessor
from os import getenv
from transformers import BitsAndBytesConfig

# 4-Bit-Quantisierung konfigurieren
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

# 1. Modell und Tokenizer laden
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
HF_TOKEN = getenv("HF_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=HF_TOKEN,
    clean_up_tokenization_spaces=False
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=HF_TOKEN,
    torch_dtype="auto",
    device_map="mps",
    quantization_config=quantization_config
)

# 2. SynthID LogitsProcessor konfigurieren
# Ein geheimer Schlüssel (keys) steuert die mathematische Pseudozufallsfunktion
synthid_processor = SynthIDTextWatermarkLogitsProcessor(
    keys=[1234, 5678, 9012],  # Pseudozufalls-Schlüssel
    sampling_table_size=1024,  # Größe der Hash-Tabelle für Logit-Shift
    ngram_len=5,
    sampling_table_seed=42,  # Seed für Determinismus
    context_history_size=5,  # Entspricht der Kontextlänge (ngram_len)
    device="mps"
)

# In die LogitsProcessorList von Hugging Face einreihen
logits_processor = LogitsProcessorList([synthid_processor])

# 3. Input vorbereiten
prompt = "Erkläre kurz, was ein Quantencomputer ist."
messages = [{"role": "user", "content": prompt}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

# 4. Text generieren (mit eingebettetem Wasserzeichen)
output_ids = model.generate(
    **inputs,  # <--- Hier mit ** entpacken
    max_new_tokens=400,
    do_sample=True,
    temperature=0.7,
    logits_processor=logits_processor,
)

# 5. Output dekodieren
input_length = inputs["input_ids"].shape[1]
generated_text = tokenizer.decode(
    output_ids[0][input_length:], skip_special_tokens=True
)

print("Generierter Text (mit SynthID Wasserzeichen):\n")
print(generated_text)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Generierter Text (mit SynthID Wasserzeichen):

Ein Quantencomputer ist ein Computer, der die Prinzipien der Quantenmechanik verwendet, um bestimmte Berechnungen viel effizienter und schneller durchzuführen als herkömmliche klassische Computer. Er basiert auf der Quantenmechanik, die die physikalischen Eigenschaften von Teilchen wie Elektronen und Photonen beschreibt.

Ein Quantencomputer verwendet sogenannte Quantenbits (Qubits), die eine Kombination aus herkömmlichen Bit und Quantenmechanik sind. Diese Qubits können mehrere Zustände gleichzeitig annehmen (Superposition) und in einer einzigen Operation viele Informationen verarbeiten.

Der Vorteil eines Quantencomputers liegt darin, dass er bestimmte Berechnungen in einem einzigen Schritt lösen kann, die bei klassischen Computern viele Schritte erfordern würden. Dies könnte beispielsweise bei Kryptographie, Simulierungen von physikalischen Systemen oder bei der Lösung komplexer algebraischer Gleichungen helfen.

Es ist jedoch wichtig z